In [ ]:
import requests
import csv
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re

CSV_FILE = "guardian.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.theguardian.com/",
}

def get_article_links(base_url):
    """Get Guardian article links that have a /YYYY/ in the URL."""
    try:
        resp = requests.get(base_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        links = set()
        year_pattern = re.compile(r"/\d{4}/")  # match /2023/, /2019/, etc.

        for a in soup.find_all("a", href=True):
            href = a["href"]
            if year_pattern.search(href) :
                full_url = urljoin("https://www.theguardian.com", href)
                if and "/news/" in full_url:
                    links.add(full_url)

        return sorted(links)

    except Exception as e:
        print(f"Error fetching {base_url}: {e}")
        return []


def scrape_article_guardian(url):
    """Scrape main Guardian article text from div#maincontent."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        main_div = soup.find("div", id="maincontent")
        if not main_div:
            return "", "", ""

        paragraphs = []
        for elem in main_div.find_all(["p", "li"], recursive=True):
            text = elem.get_text(strip=True)
            if text:
                paragraphs.append(text)

        title_tag = soup.find("h1")
        title_text = title_tag.get_text(strip=True) if title_tag else ""

        time_tag = soup.find("time")
        published_date = time_tag.get("datetime") if time_tag else ""

        return title_text, published_date, "\n".join(paragraphs)

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "", "", ""


def load_existing_links():
    """Load URLs already in CSV to skip duplicates."""
    if not os.path.exists(CSV_FILE):
        return set()
    with open(CSV_FILE, newline='', encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {row["url"] for row in reader}


def save_articles(articles):
    """Save scraped articles to CSV."""
    file_exists = os.path.exists(CSV_FILE)
    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["title", "url", "published_date", "content"]
        )
        if not file_exists:
            writer.writeheader()
        for article in articles:
            writer.writerow(article)


if __name__ == "__main__":
    base_urls = [
        "https://www.theguardian.com/international",
        "https://www.theguardian.com/uk/technology"
    ]

    existing_links = load_existing_links()

    all_links = set()
    for url in base_urls:
        section_links = get_article_links(url)
        print(f"Found {len(section_links)} from {url}")
        all_links.update(section_links)
        time.sleep(5)

    print(f"Total unique candidate articles: {len(all_links)}")

    scraped_data = []
    i = 0
    for link in all_links:
        if link in existing_links:
            print(f"Skipping (already scraped): {link}")
            continue
        time.sleep(10)
        print(f"Scraping: {link}")
        title, published_date, content = scrape_article_guardian(link)
        if content:
            scraped_data.append({
                "title": title,
                "url": link,
                "published_date": published_date,
                "content": content
            })
        i += 1
        if i>=5:
            break

    if scraped_data:
        save_articles(scraped_data)
    print("Done.")


Found 140 from https://www.theguardian.com/international
Found 48 from https://www.theguardian.com/uk/technology
Total unique candidate articles: 170
Scraping: https://www.theguardian.com/technology/2025/aug/08/smartwatches-offer-little-insight-into-stress-levels-researchers-find
Scraping: https://www.theguardian.com/technology/2025/jul/23/samsung-galaxy-z-fold-7-android-review-thinner-lighter-better-folding
Scraping: https://www.theguardian.com/travel/2025/aug/10/a-taste-of-swedish-island-life-but-on-the-mainland-near-stockholm
Scraping: https://www.theguardian.com/lifeandstyle/2025/aug/10/this-is-how-we-do-it-our-first-sexual-experience-was-like-everyones-bad-we-were-so-awkward
Scraping: https://www.theguardian.com/world/2025/aug/10/israel-plan-to-occupy-gaza-deepens-rift-between-government-and-idf
Done.
